# Example EDA - All Datasets
not done yet

# Setup & Imports

In [ ]:
print("Loading Libraries")
# Core Libraries
import pandas as pd # excel tools
import numpy as np # math
print("Done loading")

# Load Data from Excel Sheet
Loading from the Sheet and organising it through their sheets

In [ ]:
#reading from excel files
print('Loading sheets')
print('clinical')
# clinical data
clincal_sheet = pd.read_excel('../RA/RA_MAP_Clinical_Figshare_17_5_21.xlsx', sheet_name=None)
print('protogen')
# protein analysis
protogen_sheet = pd.read_excel('../RA/Protogen_RA_MAP_16_05_21.xlsx', sheet_name=None)
print('somascan')
# ra-responder and non responder
somascan_sheet = pd.read_excel('../RA/13322471/SOMASCAN_RA-Map_figshare_17_11_20.xlsx', sheet_name=None)
# all_sheets at once just in case its useful
print('Indexing sheets')
all_sheets = {
    'clinical': clincal_sheet,
    'protogen': protogen_sheet,
    'somascan': somascan_sheet
}
print('Done')

# Structure
Take a look at the structure and initialize variables for them

In [ ]:
#print all sheetnames
print(clincal_sheet.keys())
print(protogen_sheet.keys())
print(somascan_sheet.keys())


In [ ]:
# clinical
df_clinical = clincal_sheet['OpenPseudonymised_RA_MAP_Clinic']
df_steroids = clincal_sheet['intramuscular steroids']
df_meds = clincal_sheet['RA Meds']
df_glossary = clincal_sheet['Glossary'] 
# protogen/proteins
df_Samples = protogen_sheet['LIS_PG665-P01 RA MAP Samples Ex']
df_Samples_Annotation = protogen_sheet['Sample annotation']
# somascan
df_expMatrix = somascan_sheet['expression matrix']
df_sampMatrix = somascan_sheet['sample matrix']

for source_name, sheets in all_sheets.items():
    print(f'=== {source_name} ===')
    for sheet_name, df in sheets.items():
        print(f'  {sheet_name}: {df.shape[0]} rows x {df.shape[1]} columns')
    print()

---
## Taking sample
take a sample of a responder 

In [ ]:
filtered_df_sampMatrix = df_sampMatrix[df_sampMatrix['SampleGroup'] == 'RA-responder']
#filtered_df_sampMatrix = df_sampMatrix[df_sampMatrix['SampleGroup'] == 'RA-nonresp']
display(filtered_df_sampMatrix.head()) # taking Patient_ID TAC1000 since it is a RA-responder at Baseline (BL)
#display(df_sampMatrix.head())
# settings so it isnt truncated anymore 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
df_clinical.head()
display(df_clinical.head())
df_clinical['DAS28.3M'] = pd.to_numeric(df_clinical['DAS28.3M'], errors='coerce')
with pd.option_context('display.max_rows', None):
    print(df_clinical.dtypes.T)

In [ ]:
display(df_Samples.dtypes)
display(df_Samples.head())

# Journey
Map out all Data related to Patient **TAC1000**


In [ ]:
# First look at all the Fields used
df_glossary

In [ ]:
# quick look at all data
df_clinical[df_clinical['Patient_ID'] == 'TAC1001'].T

In [ ]:
df_sampMatrix[df_sampMatrix['Patient_ID'] == 'TAC1001'].T


In [ ]:
patient = df_clinical[df_clinical['Patient_ID'] == 'TAC1001']
# after a quick view we can establish that only these values are needed for now
columns_needed = [
    'Patient_ID', 'Digest', 'AGE', 'GENDER',
    'ACPA.POSITIVE', 'RHUEMATOID.FACTOR',
    'DAS28.0M', 'DAS28.3M', 'DAS28.6M', 'DAS28.9M', 'DAS28.12M', 'DAS28.18M',
    'Remission month'#, '4. Has the patient received a steroid injection?'
]
display(patient[columns_needed].T)
display(patient[columns_needed].dtypes)
patient


In [ ]:
remission_at_6M = df_clinical[df_clinical['Remission month']  == 6]
remission_at_12M = df_clinical[df_clinical['Remission month'] == 12]
columns_needed = [
    'Patient_ID', 'Digest', 'AGE', 'GENDER',
    'ACPA.POSITIVE', 'RHUEMATOID.FACTOR',
    'Remission month'
]

display(remission_at_6M[columns_needed])
display(remission_at_12M[columns_needed])

In [ ]:
df_clinical['Remission month'].dtype
#df_clinical['Remission month'] = pd.to_numeric(df_clinical['Remission month'], errors='coerce')
df_clinical = df_clinical.dropna(subset=['Remission month'])
df_clinical['Remission month'] = df_clinical['Remission month'].astype('Int8')
df.clinical['Remmission month'].dtype

---
# TAC1000

Digest = B1895B6B8F7B325B977C180A25C74D079EFB5EB306693DD2812816793CB20882
Her ACPA and RA-Factor are both positive
DAS at BL was high, 7.27, but calmed down at 12M, 1.13
Remission Month was achieved at 3
---
Moving to the medications and steroids she was given

In [ ]:
df_meds[df_meds['Digest'] == '59749E6B6785A9983100DA76AB0AE9484B52B2422AB3622294921491FAF71F6E']

In [ ]:
display(df_steroids[df_steroids['Digest'] == '59749E6B6785A9983100DA76AB0AE9484B52B2422AB3622294921491FAF71F6E'].T)
display(df_steroids.dtypes)

# Medications & Steroids
Medications show very good signs, since she did not need any intramuscular steroids
---
Moving on to her protein analysis to know how her immune system reacted to what protein at what point in time
High values mean more antibodies were created for the protein 

In [ ]:
#df_Samples[['ProteinID', 'Gene Symbol', 'Gene Name', 'TAC1001_M6']].sort_values('TAC1001_M6', ascending=False)
df_clinical[['HB.0M', 'HB.6M']].sample(10).sort_values(['HB.0M', 'HB.6M'], ascending=False)
print((df_clinical['HB.0M'] < 30).sum())


In [ ]:
df_clinical['HB.0M'] = pd.to_numeric(df_clinical['HB.0M'], errors='coerce')
df_clinical['HB.6M'] = pd.to_numeric(df_clinical['HB.6M'], errors='coerce')
print('=== HB.0M ===')
print(f'Min: {df_clinical["HB.0M"].min()}')
print(f'Max: {df_clinical["HB.0M"].max()}')
print(f'Werte unter 30: {(df_clinical["HB.0M"] < 30).sum()}')
print(f'Werte über 30: {(df_clinical["HB.0M"] >= 30).sum()}')
print()
print('Alle Werte unter 30:')
print(df_clinical['HB.0M'][df_clinical['HB.0M'] < 30].sort_values().values)
print()
print('=== HB.6M ===')
print(f'Min: {df_clinical["HB.6M"].min()}')
print(f'Max: {df_clinical["HB.6M"].max()}')
print(f'Werte unter 30: {(df_clinical["HB.6M"] < 30).sum()}')
print(f'Werte über 30: {(df_clinical["HB.6M"] >= 30).sum()}')
print()
print('Alle Werte unter 30:')
print(df_clinical['HB.6M'][df_clinical['HB.6M'] < 30].sort_values().values)

# Protein Analysis

---
# Somascan

In [ ]:
somascan_cols = [col for col in df_expMatrix.columns if 'TAC1001' in str(col)]
print(f'Gefundene Spalten: {somascan_cols}')
print(f'Anzahl Proteine: {df_expMatrix.shape[0]}')

# Zeige die Proteinwerte
id_cols = [df_expMatrix.columns[0]]  # Erste Spalte als Identifier
df_expMatrix[id_cols + somascan_cols]